# Chat-CE v2 — fine-tune with synthetic preference augmentation

Base: `cross-encoder/ms-marco-MiniLM-L-12-v2`  
Loss: BinaryCrossEntropyLoss  
Data: Sprint 1 hard-neg pairs + Sprint 2 synthetic preference paraphrases (Llama-3.3-70B)
Train pairs: 2430 (s1=848 + syn=1582), val pairs: 421

**Kaggle:** Add Input → `chat-ce-v2-aug` dataset. Accelerator → **GPU T4 x2**. Run All.
Beklenen süre: T4 ~5-10 dk.

In [ ]:
# 1) Setup (single GPU lock + deps)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==5.4.1', 'torch'])
import torch
print('cuda?', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) Locate data (glob works for any /kaggle/input/* layout)
from pathlib import Path
import glob
found = glob.glob('/kaggle/input/**/task2_train_pairs.jsonl', recursive=True) + glob.glob('/content/task2_train_pairs.jsonl') + glob.glob('./task2_train_pairs.jsonl')
assert found, 'train file not found. Add dataset via right panel.'
TRAIN = Path(found[0])
VAL = TRAIN.parent / 'task2_val_pairs.jsonl'
print('train:', TRAIN)
print('val:  ', VAL)

In [ ]:
# 3) Load examples
import json
from sentence_transformers import InputExample
def load(p):
    out = []
    for line in open(p):
        r = json.loads(line)
        out.append(InputExample(texts=[r['q'], r['doc']], label=float(r['label'])))
    return out
train_ex = load(TRAIN); val_ex = load(VAL)
print(f'train={len(train_ex)} val={len(val_ex)}')

In [ ]:
# 4) Train
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader
BASE = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
OUT = './chat-ce-v2-20260516'
EPOCHS = 3; BATCH = 32; LR = 2e-5; MAX_LEN = 384
model = CrossEncoder(BASE, num_labels=1, max_length=MAX_LEN)
loader = DataLoader(train_ex, shuffle=True, batch_size=BATCH)
val_pairs = [[ex.texts[0], ex.texts[1]] for ex in val_ex]
val_labels = [int(ex.label) for ex in val_ex]
evaluator = CEBinaryClassificationEvaluator(val_pairs, val_labels, name='val')
warmup = int(0.1 * len(loader) * EPOCHS)
print(f'epochs={EPOCHS} batch={BATCH} lr={LR} steps/epoch={len(loader)} warmup={warmup}')
model.fit(train_dataloader=loader, evaluator=evaluator, epochs=EPOCHS, warmup_steps=warmup,
          optimizer_params={'lr': LR}, output_path=OUT, save_best_model=False, show_progress_bar=True)
# manual save (Sprint 1 bug: save_best_model with new Trainer didn't write)
model.save(OUT)
import os; print('saved files:', sorted(os.listdir(OUT)))

In [ ]:
# 5) Val sanity
import numpy as np
from sentence_transformers import CrossEncoder as CE
best = CE(OUT, max_length=MAX_LEN)
scores = best.predict(val_pairs, batch_size=64, show_progress_bar=False)
preds = (np.array(scores) > 0.5).astype(int)
labels = np.array(val_labels)
acc = float((preds == labels).mean())
print(f'val acc (>0.5): {acc:.4f}  pos={int(labels.sum())} neg={int((1-labels).sum())}')

In [ ]:
# 6) Pack + FileLink for download
import shutil, os
shutil.make_archive('chat-ce-v2', 'zip', OUT)
print('size:', os.path.getsize('chat-ce-v2.zip')//1024, 'KB')
from IPython.display import FileLink
FileLink('chat-ce-v2.zip')